# Notebook 01 — RLM Basics with smolagents

This notebook introduces the core concepts of **Recursive Language Models (RLMs)** and shows how the implementation in `src/rlm_smolagent.py` works.

## What is an RLM?

> *Recursive Language Models are a task-agnostic inference paradigm for language models to handle near-infinite length contexts by enabling the LM to programmatically examine, decompose, and recursively call itself over its input.*
> — [Zhang et al., 2025](https://arxiv.org/abs/2512.24601)

Traditional LLM call:
```python
response = llm.completion(prompt)   # context window is a hard limit
```

RLM call:
```python
response = rlm.completion(prompt)   # the model can split & call itself
```

Internally the model runs inside a **Python REPL** (provided by smolagents' `CodeAgent`).  It can:
1. Inspect and split its input.
2. Call `rlm_call(sub_prompt)` for each piece.
3. Aggregate the child responses in Python code.

## Architecture

```
 ┌─────────────────────────────────────────────┐
 │  RLMAgent.completion(prompt)                │
 │                                             │
 │  smolagents CodeAgent  ◄──► REPL (Python)   │
 │      │                        │             │
 │      │  rlm_call(sub_prompt)  │             │
 │      └────────────────────────┘             │
 │              │                              │
 │      child RLMAgent (depth+1)               │
 │              │                              │
 │           … until max_depth                 │
 │           (plain LLM call)                  │
 └─────────────────────────────────────────────┘
```

## LLM backend

The container is configured to reach the **host llama.cpp server** via the `LLAMA_BASE_URL` environment variable (default: `http://host-gateway:8080/v1`).  The llama.cpp server exposes an OpenAI-compatible REST API, so the same `openai` Python client is used throughout.

## 1. Configuration

In [ ]:
import os
import sys

# Make the src/ module importable from the notebook
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

# Connection settings — read from environment variables set by docker-compose,
# with sensible defaults for manual runs.
LLAMA_BASE_URL = os.environ.get('LLAMA_BASE_URL', 'http://host-gateway:8080/v1')
LLAMA_MODEL    = os.environ.get('LLAMA_MODEL',    'local-model')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed')

print(f'Backend URL : {LLAMA_BASE_URL}')
print(f'Model name  : {LLAMA_MODEL}')

## 2. Verify the llama.cpp server is reachable

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=LLAMA_BASE_URL, api_key=OPENAI_API_KEY)

try:
    models = client.models.list()
    print('✅ Server is reachable.  Available models:')
    for m in models.data:
        print(f'   • {m.id}')
except Exception as exc:
    print(f'❌ Could not reach the server: {exc}')
    print('   Make sure the llama.cpp server is running on the host and'
          ' LLAMA_BASE_URL points to it.')

## 3. Plain LLM completion (baseline)

In [ ]:
resp = client.chat.completions.create(
    model=LLAMA_MODEL,
    messages=[{"role": "user", "content": "What is 2 + 2? Answer with just the number."}],
)
print('Plain LLM response:', resp.choices[0].message.content)

## 4. Import RLMAgent

In [ ]:
from rlm_smolagent import RLMAgent

agent = RLMAgent(
    base_url=LLAMA_BASE_URL,
    model_name=LLAMA_MODEL,
    api_key=OPENAI_API_KEY,
    max_depth=2,     # allow up to 2 levels of recursion
    max_steps=8,     # max REPL cycles per depth level
    verbose=True,    # print agent reasoning
)
print('RLMAgent ready.')

## 5. Simple RLM completion

A short task that may not require recursion but demonstrates the agent loop.

In [ ]:
result = agent.completion(
    "List the first 5 prime numbers, one per line."
)

print('\n=== Final Response ===')
print(result.response)

## 6. Inspect the call tree (metadata)

In [ ]:
import json

print(json.dumps(result.metadata, indent=2))

## 7. Recursive task — divide and conquer summation

We give the agent a list of 50 numbers and ask it to sum them.
A smart RLM will split the list, delegate each half via `rlm_call`, and
combine the partial sums — demonstrating real recursive decomposition.

In [ ]:
import random
random.seed(42)

numbers = [random.randint(1, 100) for _ in range(50)]
expected = sum(numbers)

prompt = (
    "You have a list of numbers. "
    "Split it into two halves, use rlm_call() to sum each half recursively, "
    "then add the two partial sums together.\n\n"
    f"Numbers: {numbers}"
)

result = agent.completion(prompt)

print(f'Expected sum : {expected}')
print(f'RLM response : {result.response}')

In [ ]:
# Inspect the recursive call tree
print(json.dumps(result.metadata, indent=2))